In [1]:
!pip install segmentation-models-pytorch -q
!pip install scikit-image -q
!pip install pandas -q
!pip install scipy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 2.4 MB/s eta 0:00:00


In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd

import segmentation_models_pytorch as smp

from tqdm import tqdm
from skimage.morphology import skeletonize

In [3]:
from scipy.signal import convolve2d

def count_branch_end_points(skeleton):

    kernel = np.array([
        [1,1,1],
        [1,10,1],
        [1,1,1]
    ])

    neighbors = convolve2d(
        skeleton.astype(np.uint8),
        kernel,
        mode="same"
    )

    branch_points = np.sum(neighbors >= 13)

    end_points = np.sum(neighbors == 11)

    return int(branch_points), int(end_points)

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
import os

PROJECT_DIR = "/content/drive/MyDrive/RetinalStrokeProject"

MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "unet_vessel_segmentation.pth"
)

DATA_DIR = "/content/drive/MyDrive/dataset_final"

TEST_IMAGE_DIR = os.path.join(
    DATA_DIR,
    "test/images"
)

print("Model:", MODEL_PATH)
print("Test Images:", TEST_IMAGE_DIR)

Model: /content/drive/MyDrive/RetinalStrokeProject/models/unet_vessel_segmentation.pth
Test Images: /content/drive/MyDrive/dataset_final/test/images


In [6]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=1,
    classes=1
)

model.load_state_dict(
    torch.load(
        MODEL_PATH,
        map_location=device
    )
)

model = model.to(device)

model.eval()

print("Model Loaded Successfully")

Model Loaded Successfully


In [7]:
def preprocess_image(img_path):

    img = cv2.imread(
        img_path,
        cv2.IMREAD_GRAYSCALE
    )

    img = cv2.resize(
        img,
        (512,512)
    )

    img = img / 255.0

    img = np.expand_dims(
        img,
        axis=0
    )

    img = np.expand_dims(
        img,
        axis=0
    )

    img = torch.tensor(
        img,
        dtype=torch.float32
    )

    return img

In [8]:
def vessel_density(mask):

    vessel_pixels = np.sum(mask)

    total_pixels = mask.shape[0] * mask.shape[1]

    return vessel_pixels / total_pixels

In [9]:
def vessel_length(mask):

    skeleton = skeletonize(mask)

    return np.sum(skeleton)

In [13]:
features = []

image_files = sorted(
    os.listdir(TEST_IMAGE_DIR)
)

for image_name in tqdm(image_files):

    image_path = os.path.join(
        TEST_IMAGE_DIR,
        image_name
    )

    image = preprocess_image(
        image_path
    )

    image = image.to(device)

    with torch.no_grad():

        pred = model(image)

        pred = torch.sigmoid(pred)

        pred = pred.squeeze()

        pred = pred.cpu().numpy()

    pred_mask = (pred > 0.5).astype(np.uint8)

    # Feature 1: Vessel Density
    density = vessel_density(pred_mask)

    # Feature 2: Vessel Length
    skeleton = skeletonize(pred_mask > 0)

    length = np.sum(skeleton)

    # Feature 3 & 4: Branch Points and End Points
    branch_points, end_points = count_branch_end_points(
        skeleton
    )
    tortuosity = vessel_tortuosity(
    skeleton
)

    features.append([
    image_name,
    density,
    length,
    branch_points,
    end_points,
    tortuosity
])

100%|██████████| 14/14 [00:34<00:00,  2.45s/it]


In [14]:
df = pd.DataFrame(
    features,
    columns=[
    "image_name",
    "vessel_density",
    "vessel_length",
    "branch_points",
    "end_points",
    "tortuosity"
]
)

df.head()

,image_name,vessel_density,vessel_length,branch_points,end_points,tortuosity
0,09_dr.JPG,0.030357,2895,49,405,4.740268
1,09_g.png,0.059147,5015,150,302,9.914359
2,13_h.png,0.072987,5867,177,461,11.049043
3,14_h.png,0.069824,5628,188,355,11.064258
4,14_test.png,0.069733,4721,243,240,10.436724


In [15]:
FEATURE_PATH = os.path.join(
    PROJECT_DIR,
    "features.csv"
)

df.to_csv(
    FEATURE_PATH,
    index=False
)

print("Saved:", FEATURE_PATH)

Saved: /content/drive/MyDrive/RetinalStrokeProject/features.csv


In [12]:
def vessel_tortuosity(skeleton):

    coords = np.column_stack(
        np.where(skeleton > 0)
    )

    if len(coords) < 2:
        return 0

    vessel_length = len(coords)

    start = coords[0]
    end = coords[-1]

    straight_distance = np.linalg.norm(
        start - end
    )

    if straight_distance == 0:
        return 0

    tortuosity = vessel_length / straight_distance

    return float(tortuosity)